<a href="https://colab.research.google.com/github/julianovale/geoestatistica/blob/main/000_MDT_RioClaro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
! pip install ezdxf
! pip install rasterio
! pip install plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 19.4 MB/s  0:00:00


In [4]:
import re
import numpy as np
import ezdxf
from scipy.interpolate import griddata
import rasterio
from rasterio.transform import from_origin
import plotly.graph_objects as go
from google.colab import drive

# ==========================================
# 1. MONTAR GOOGLE DRIVE E CONFIGURAR CAMINHOS
# ==========================================
drive.mount("/content/drive")

PASTA_DRIVE = "/content/drive/MyDrive/Backup Computador Unesp Rio Claro/Rio Claro 1_50 000"
CAMINHO_DXF = f"{PASTA_DRIVE}/RClaro.dxf"
CAMINHO_TIF = f"{PASTA_DRIVE}/mdt_rio_claro_50k.tif"
CAMINHO_STL = f"{PASTA_DRIVE}/mdt_rio_claro_50k.stl"

CRS_EPSG = "EPSG:22523"  # Corrego Alegre / UTM zone 23S (Esferoide Internacional)[cite: 1]
RES_GRADE = 25.0         # Resolucao em metros (equidistancia das curvas = 20 m)[cite: 1]
EXAGERO_Z = 2.5          # Exagero vertical apenas para visualizacao 3D

# ==========================================
# 2. LER DXF E EXTRAIR COTAS
# ==========================================
doc = ezdxf.readfile(CAMINHO_DXF)
msp = doc.modelspace()

layers = sorted({entity.dxf.layer for entity in msp})
print("Camadas encontradas:", layers)

def extrair_elevacao_dxf(entity):
    """Extrai elevacao numerica segura de entity.dxf.elevation (pode ser float ou Vec3)."""
    if not entity.dxf.hasattr("elevation"):
        return None
    elev = entity.dxf.elevation
    if isinstance(elev, (int, float, np.number)):
        val = float(elev)
        return val if np.isfinite(val) and val != 0 else None
    try:
        # Em alguns casos no ezdxf, elevation e Vec3(x, y, z)
        val = float(elev.z)
        return val if np.isfinite(val) and val != 0 else None
    except AttributeError:
        try:
            val = float(elev)
            return val if np.isfinite(val) and val != 0 else None
        except (TypeError, ValueError):
            return None

def cota_da_camada(layer_name):
    """Converte nome da camada em cota quando a camada e numerica (ex.: '500' -> 500.0)."""
    if not layer_name:
        return None
    limpo = layer_name.strip().replace(",", ".")
    if re.fullmatch(r"-?\d+(?:\.\d+)?", limpo):
        val = float(limpo)
        if 0 < val < 3000:
            return val
    return None

def densificar_segmento(p1, p2, passo=25.0):
    """Interpola pontos intermediarios entre dois vertices para evitar 'degraus'."""
    dist = np.hypot(p2[0] - p1[0], p2[1] - p1[1])
    if dist <= passo or dist <= 0:
        return [p1, p2]
    n = int(np.ceil(dist / passo))
    xs = np.linspace(p1[0], p2[0], n + 1)
    ys = np.linspace(p1[1], p2[1], n + 1)
    return list(zip(xs, ys))

def densificar_lista(pontos, passo=25.0):
    if len(pontos) < 2:
        return pontos
    out = []
    for i in range(len(pontos) - 1):
        trecho = densificar_segmento(pontos[i], pontos[i + 1], passo=passo)
        if i > 0:
            trecho = trecho[1:]
        out.extend(trecho)
    return out

xs, ys, zs = [], [], []

for entity in msp:
    tipo = entity.dxftype()
    layer = entity.dxf.layer
    z_layer = cota_da_camada(layer)
    z_elev = extrair_elevacao_dxf(entity)

    if tipo == "POINT":
        loc = entity.dxf.location
        z = loc.z if (loc.z != 0 and np.isfinite(loc.z)) else (z_elev if z_elev is not None else z_layer)
        if z is not None:
            xs.append(loc.x); ys.append(loc.y); zs.append(float(z))

    elif tipo == "LWPOLYLINE":
        z = z_elev if z_elev is not None else z_layer
        if z is not None:
            pts_2d = [(p[0], p[1]) for p in entity.get_points()]
            for x, y in densificar_lista(pts_2d, passo=RES_GRADE):
                xs.append(x); ys.append(y); zs.append(float(z))

    elif tipo == "POLYLINE":
        z_verts = [
            float(v.dxf.location.z)
            for v in entity.vertices
            if v.dxf.location.z != 0 and np.isfinite(v.dxf.location.z)
        ]
        z_final = z_verts[0] if z_verts else (z_elev if z_elev is not None else z_layer)
        if z_final is not None:
            pts_2d = [(v.dxf.location.x, v.dxf.location.y) for v in entity.vertices]
            for x, y in densificar_lista(pts_2d, passo=RES_GRADE):
                xs.append(x); ys.append(y); zs.append(float(z_final))

    elif tipo == "LINE":
        s, e = entity.dxf.start, entity.dxf.end
        z_base = z_elev if z_elev is not None else z_layer
        z_s = s.z if (s.z != 0 and np.isfinite(s.z)) else z_base
        z_e = e.z if (e.z != 0 and np.isfinite(e.z)) else z_base
        if z_s is not None and z_e is not None:
            pts_2d = densificar_segmento((s.x, s.y), (e.x, e.y), passo=RES_GRADE)
            z_interp = np.linspace(float(z_s), float(z_e), len(pts_2d))
            for (x, y), z in zip(pts_2d, z_interp):
                xs.append(x); ys.append(y); zs.append(float(z))

xs = np.array(xs, dtype=np.float64)
ys = np.array(ys, dtype=np.float64)
zs = np.array(zs, dtype=np.float64)

valido = np.isfinite(xs) & np.isfinite(ys) & np.isfinite(zs)
xs, ys, zs = xs[valido], ys[valido], zs[valido]

print(f"Amostras extraidas: {len(zs)}")
if len(zs) < 4:
    raise RuntimeError("Poucos pontos extraidos. Verifique como os pontos da camada PONTOS estao armazenados.")

print(f"Z extraido min / max: {zs.min():.1f} m / {zs.max():.1f} m")

# ==========================================
# 3. INTERPOLAR GRADE REGULAR (MDT)
# ==========================================
x_min, x_max = xs.min(), xs.max()
y_min, y_max = ys.min(), ys.max()

grid_x, grid_y = np.meshgrid(
    np.arange(x_min, x_max + RES_GRADE, RES_GRADE),
    np.arange(y_min, y_max + RES_GRADE, RES_GRADE)
)

grid_z = griddata((xs, ys), zs, (grid_x, grid_y), method="cubic")

mask_nan = np.isnan(grid_z)
if np.any(mask_nan):
    grid_z[mask_nan] = griddata(
        (xs, ys), zs, (grid_x[mask_nan], grid_y[mask_nan]), method="nearest"
    )

print(f"Grade gerada: {grid_z.shape[0]} linhas x {grid_z.shape[1]} colunas | Z final: {grid_z.min():.1f} m a {grid_z.max():.1f} m")

# ==========================================
# 4. SALVAR GEOTIFF (.TIF) NO DRIVE
# ==========================================
grid_z_top_down = np.flipud(grid_z.astype(np.float32))
transform = from_origin(x_min, y_max, RES_GRADE, RES_GRADE)

with rasterio.open(
    CAMINHO_TIF,
    "w",
    driver="GTiff",
    height=grid_z_top_down.shape[0],
    width=grid_z_top_down.shape[1],
    count=1,
    dtype="float32",
    crs=CRS_EPSG,
    transform=transform,
    compress="lzw",
) as dst:
    dst.write(grid_z_top_down, 1)

print("GeoTIFF salvo em:", CAMINHO_TIF)

# ==========================================
# 5. SALVAR MALHA 3D (.STL) NO DRIVE
# ==========================================
def salvar_stl_ascii(gx, gy, gz, caminho_stl):
    rows, cols = gz.shape
    with open(caminho_stl, "w") as f:
        f.write("solid mdt_rio_claro\n")
        for r in range(rows - 1):
            for c in range(cols - 1):
                p00 = (gx[r, c], gy[r, c], gz[r, c])
                p01 = (gx[r, c + 1], gy[r, c + 1], gz[r, c + 1])
                p10 = (gx[r + 1, c], gy[r + 1, c], gz[r + 1, c])
                p11 = (gx[r + 1, c + 1], gy[r + 1, c + 1], gz[r + 1, c + 1])

                for a, b, d in [(p00, p10, p11), (p00, p11, p01)]:
                    u = np.subtract(b, a)
                    v = np.subtract(d, a)
                    n = np.cross(u, v)
                    norm = np.linalg.norm(n)
                    n = n / norm if norm > 0 else np.array([0.0, 0.0, 1.0])
                    f.write(f"  facet normal {n[0]:.6e} {n[1]:.6e} {n[2]:.6e}\n")
                    f.write("    outer loop\n")
                    f.write(f"      vertex {a[0]:.6f} {a[1]:.6f} {a[2]:.6f}\n")
                    f.write(f"      vertex {b[0]:.6f} {b[1]:.6f} {b[2]:.6f}\n")
                    f.write(f"      vertex {d[0]:.6f} {d[1]:.6f} {d[2]:.6f}\n")
                    f.write("    endloop\n")
                    f.write("  endfacet\n")
        f.write("endsolid mdt_rio_claro\n")

salvar_stl_ascii(grid_x, grid_y, grid_z * EXAGERO_Z, CAMINHO_STL)
print("Malha STL salva em:", CAMINHO_STL)

# ==========================================
# 6. VISUALIZACAO 3D INTERATIVA NO COLAB
# ==========================================
passo_vis = max(1, max(grid_z.shape) // 350)

fig = go.Figure(
    data=[
        go.Surface(
            x=grid_x[::passo_vis, ::passo_vis],
            y=grid_y[::passo_vis, ::passo_vis],
            z=grid_z[::passo_vis, ::passo_vis],
            colorscale="Earth",
            colorbar=dict(title="Altitude (m)"),
        )
    ]
)

fig.update_layout(
    title="MDT 3D - Folha Rio Claro (1:50.000)",
    scene=dict(
        xaxis_title="UTM E (m)",
        yaxis_title="UTM N (m)",
        zaxis_title="Altitude (m)",
        aspectratio=dict(x=1, y=1, z=0.25),
    ),
    height=700,
)
fig.show()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Camadas encontradas: ['500', '520', '540', '560', '580', '600', '620', '640', '660', '680', '700', '720', '740', '760', '780', '800', '820', '840', '860', 'PONTOS']
Amostras extraidas: 184159
Z extraido min / max: 500.0 m / 861.0 m
Grade gerada: 1127 linhas x 1051 colunas | Z final: 464.9 m a 1033.5 m
GeoTIFF salvo em: /content/drive/MyDrive/Backup Computador Unesp Rio Claro/Rio Claro 1_50 000/mdt_rio_claro_50k.tif
Malha STL salva em: /content/drive/MyDrive/Backup Computador Unesp Rio Claro/Rio Claro 1_50 000/mdt_rio_claro_50k.stl


In [7]:
import re
import numpy as np
import ezdxf
from scipy.interpolate import griddata
from scipy.ndimage import gaussian_filter
import rasterio
from rasterio.transform import from_origin
import plotly.graph_objects as go
from google.colab import drive

# ==========================================
# 1. DRIVE E PARAMETROS DE LIMPEZA
# ==========================================
drive.mount("/content/drive")

PASTA_DRIVE = "/content/drive/MyDrive/Backup Computador Unesp Rio Claro/Rio Claro 1_50 000"
CAMINHO_DXF = f"{PASTA_DRIVE}/RClaro.dxf"
CAMINHO_TIF_LIMPO = f"{PASTA_DRIVE}/mdt_rio_claro_50k_limpo.tif"

CRS_EPSG = "EPSG:22523"   # Corrego Alegre / UTM zone 23S[cite: 1]
RES_GRADE = 25.0          # Resolucao em metros (curvas a cada 20 m)[cite: 1]
Z_MIN_VALIDO = 460.0      # Limite inferior esperado para a folha[cite: 2]
Z_MAX_VALIDO = 880.0      # Limite superior esperado (camadas vao ate 860)[cite: 2]
SIGMA_SUAVIZACAO = 1.0    # 0 = sem suavizacao; 0.8 a 1.5 = suavizacao leve

# ==========================================
# 2. EXTRAIR DADOS E DIAGNOSTICAR EXTREMOS
# ==========================================
doc = ezdxf.readfile(CAMINHO_DXF)
msp = doc.modelspace()

def extrair_elevacao_dxf(entity):
    if not entity.dxf.hasattr("elevation"):
        return None
    elev = entity.dxf.elevation
    if isinstance(elev, (int, float, np.number)):
        val = float(elev)
        return val if np.isfinite(val) and val != 0 else None
    try:
        val = float(elev.z)
        return val if np.isfinite(val) and val != 0 else None
    except AttributeError:
        try:
            val = float(elev)
            return val if np.isfinite(val) and val != 0 else None
        except (TypeError, ValueError):
            return None

def cota_da_camada(layer_name):
    if not layer_name:
        return None
    limpo = layer_name.strip().replace(",", ".")
    if re.fullmatch(r"-?\d+(?:\.\d+)?", limpo):
        val = float(limpo)
        if 0 < val < 3000:
            return val
    return None

def densificar_segmento(p1, p2, passo=25.0):
    dist = np.hypot(p2[0] - p1[0], p2[1] - p1[1])
    if dist <= passo or dist <= 0:
        return [p1, p2]
    n = int(np.ceil(dist / passo))
    xs = np.linspace(p1[0], p2[0], n + 1)
    ys = np.linspace(p1[1], p2[1], n + 1)
    return list(zip(xs, ys))

def densificar_lista(pontos, passo=25.0):
    if len(pontos) < 2:
        return pontos
    out = []
    for i in range(len(pontos) - 1):
        trecho = densificar_segmento(pontos[i], pontos[i + 1], passo=passo)
        if i > 0:
            trecho = trecho[1:]
        out.extend(trecho)
    return out

registros = []  # (x, y, z, layer, tipo)

for entity in msp:
    tipo = entity.dxftype()
    layer = entity.dxf.layer
    z_layer = cota_da_camada(layer)
    z_elev = extrair_elevacao_dxf(entity)

    if tipo == "POINT":
        loc = entity.dxf.location
        z = loc.z if (loc.z != 0 and np.isfinite(loc.z)) else (z_elev if z_elev is not None else z_layer)
        if z is not None:
            registros.append((loc.x, loc.y, float(z), layer, tipo))

    elif tipo == "LWPOLYLINE":
        z = z_elev if z_elev is not None else z_layer
        if z is not None:
            pts_2d = [(p[0], p[1]) for p in entity.get_points()]
            for x, y in densificar_lista(pts_2d, passo=RES_GRADE):
                registros.append((x, y, float(z), layer, tipo))

    elif tipo == "POLYLINE":
        z_verts = [
            float(v.dxf.location.z)
            for v in entity.vertices
            if v.dxf.location.z != 0 and np.isfinite(v.dxf.location.z)
        ]
        z_final = z_verts[0] if z_verts else (z_elev if z_elev is not None else z_layer)
        if z_final is not None:
            pts_2d = [(v.dxf.location.x, v.dxf.location.y) for v in entity.vertices]
            for x, y in densificar_lista(pts_2d, passo=RES_GRADE):
                registros.append((x, y, float(z_final), layer, tipo))

    elif tipo == "LINE":
        s, e = entity.dxf.start, entity.dxf.end
        z_base = z_elev if z_elev is not None else z_layer
        z_s = s.z if (s.z != 0 and np.isfinite(s.z)) else z_base
        z_e = e.z if (e.z != 0 and np.isfinite(e.z)) else z_base
        if z_s is not None and z_e is not None:
            pts_2d = densificar_segmento((s.x, s.y), (e.x, e.y), passo=RES_GRADE)
            z_interp = np.linspace(float(z_s), float(z_e), len(pts_2d))
            for (x, y), z in zip(pts_2d, z_interp):
                registros.append((x, y, float(z), layer, tipo))

# Inspecionar valores fora de [Z_MIN_VALIDO, Z_MAX_VALIDO]
outliers = [
    r for r in registros
    if np.isfinite(r[2]) and (r[2] < Z_MIN_VALIDO or r[2] > Z_MAX_VALIDO)
]

print(f"Total de amostras brutas: {len(registros)}")
print(f"Amostras fora de [{Z_MIN_VALIDO}, {Z_MAX_VALIDO}] m: {len(outliers)}")

# Resumo de outliers por camada e tipo
resumo_out = {}
for x, y, z, layer, tipo in outliers:
    chave = (layer, tipo)
    if chave not in resumo_out:
        resumo_out[chave] = {"qtd": 0, "z_min": z, "z_max": z, "exemplos": []}
    d = resumo_out[chave]
    d["qtd"] += 1
    d["z_min"] = min(d["z_min"], z)
    d["z_max"] = max(d["z_max"], z)
    if len(d["exemplos"]) < 3:
        d["exemplos"].append((round(x, 1), round(y, 1), round(z, 2)))

for (layer, tipo), info in sorted(resumo_out.items()):
    print(
        f"  Camada '{layer}' [{tipo}]: {info['qtd']} pts | "
        f"Z de {info['z_min']:.2f} a {info['z_max']:.2f} | ex.: {info['exemplos']}"
    )

# ==========================================
# 3. FILTRAR EXTREMOS E INTERPOLAR (LINEAR + SUAVE)
# ==========================================
filtrados = [
    (x, y, z) for (x, y, z, _, _) in registros
    if np.isfinite(x) and np.isfinite(y) and np.isfinite(z)
    and (Z_MIN_VALIDO <= z <= Z_MAX_VALIDO)
]

xs = np.array([p[0] for p in filtrados], dtype=np.float64)
ys = np.array([p[1] for p in filtrados], dtype=np.float64)
zs = np.array([p[2] for p in filtrados], dtype=np.float64)

print(f"Amostras apos filtro: {len(zs)} | Z: {zs.min():.1f} m a {zs.max():.1f} m")

x_min, x_max = xs.min(), xs.max()
y_min, y_max = ys.min(), ys.max()

grid_x, grid_y = np.meshgrid(
    np.arange(x_min, x_max + RES_GRADE, RES_GRADE),
    np.arange(y_min, y_max + RES_GRADE, RES_GRADE)
)

# Interpolacao linear evita picos/overshoots do metodo cubico
grid_z = griddata((xs, ys), zs, (grid_x, grid_y), method="linear")

mask_nan = np.isnan(grid_z)
if np.any(mask_nan):
    grid_z[mask_nan] = griddata(
        (xs, ys), zs, (grid_x[mask_nan], grid_y[mask_nan]), method="nearest"
    )

if SIGMA_SUAVIZACAO > 0:
    grid_z = gaussian_filter(grid_z, sigma=SIGMA_SUAVIZACAO)

print(f"Grade final: {grid_z.shape[0]}x{grid_z.shape[1]} | Z: {grid_z.min():.1f} m a {grid_z.max():.1f} m")

# ==========================================
# 4. SALVAR GEOTIFF LIMPO NO DRIVE
# ==========================================
grid_z_top_down = np.flipud(grid_z.astype(np.float32))
transform = from_origin(x_min, y_max, RES_GRADE, RES_GRADE)

with rasterio.open(
    CAMINHO_TIF_LIMPO,
    "w",
    driver="GTiff",
    height=grid_z_top_down.shape[0],
    width=grid_z_top_down.shape[1],
    count=1,
    dtype="float32",
    crs=CRS_EPSG,
    transform=transform,
    compress="lzw",
) as dst:
    dst.write(grid_z_top_down, 1)

print("GeoTIFF limpo salvo em:", CAMINHO_TIF_LIMPO)

# ==========================================
# 5. VISUALIZACAO 3D INTERATIVA NO COLAB
# ==========================================
passo_vis = max(1, max(grid_z.shape) // 350)

fig = go.Figure(
    data=[
        go.Surface(
            x=grid_x[::passo_vis, ::passo_vis],
            y=grid_y[::passo_vis, ::passo_vis],
            z=grid_z[::passo_vis, ::passo_vis],
            colorscale="Earth",
            colorbar=dict(title="Altitude (m)"),
        )
    ]
)

fig.update_layout(
    title="MDT 3D Lapidado - Folha Rio Claro (1:50.000)",
    scene=dict(
        xaxis_title="UTM E (m)",
        yaxis_title="UTM N (m)",
        zaxis_title="Altitude (m)",
        aspectratio=dict(x=1, y=1, z=0.1),
    ),
    height=700,
)
fig.show()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total de amostras brutas: 184159
Amostras fora de [460.0, 880.0] m: 0
Amostras apos filtro: 184159 | Z: 500.0 m a 861.0 m
Grade final: 1127x1051 | Z: 500.0 m a 860.2 m
GeoTIFF limpo salvo em: /content/drive/MyDrive/Backup Computador Unesp Rio Claro/Rio Claro 1_50 000/mdt_rio_claro_50k_limpo.tif
